In [42]:
import os
import sys

# Ścieżki Kaggle
import os
import sys

# Konfiguracja
WORKING_DIR = '/kaggle/working'
INPUT_PATH = '/kaggle/input/datasets/maciejjakubowski/full-climbing-dataset'
REPO_NAME = 'e2e-climbing-vision'
REPO_URL = f'https://github.com/jeicam3/{REPO_NAME}'
BRANCH_NAME = 'devel'

# Przejście do katalogu roboczego
os.chdir(WORKING_DIR)

# Klonowanie/aktualizacja repo
if not os.path.exists(f'{WORKING_DIR}/{REPO_NAME}'):
    print(f"Klonowanie repozytorium (branch: {BRANCH_NAME})...")
    !git clone -b {BRANCH_NAME} {REPO_URL}
else:
    print(f"Aktualizacja repozytorium (branch: {BRANCH_NAME})...")
    # Fetch i checkout na wypadek, gdybyśmy wcześniej byli na innym branchu
    !git -C {WORKING_DIR}/{REPO_NAME} fetch origin
    !git -C {WORKING_DIR}/{REPO_NAME} checkout {BRANCH_NAME}
    !git -C {WORKING_DIR}/{REPO_NAME} pull origin {BRANCH_NAME}

# Dodanie repozytorium do ścieżki Python
repo_full_path = os.path.join(WORKING_DIR, REPO_NAME)
if repo_full_path not in sys.path:
    sys.path.append(repo_full_path)

print(f"\nŚrodowisko gotowe.")

Aktualizacja repozytorium (branch: devel)...
Already on 'devel'
Your branch is up to date with 'origin/devel'.
From https://github.com/jeicam3/e2e-climbing-vision
 * branch            devel      -> FETCH_HEAD
Already up to date.

Środowisko gotowe.


In [43]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import pandas as pd
import os
import copy

# Importy z repo (upewnij się, że nazwy plików w repo są poprawne)
from models.dataset import ClimbingDataset
from models.efficientnet import get_climbing_model

# Transformacje (bez zmian)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1))
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])



CSV_PATH = f'{INPUT_PATH}/labels.csv'
IMG_DIR = f'{INPUT_PATH}/full-data/full-data'

# Zbiór testowy
test_videos = ['p3_orange', 'p3_green', 'p4_orange', 'p4_green']

df = pd.read_csv(CSV_PATH)

# Maska do zbioru testowego
val_mask = df.iloc[:, 0].str.contains('|'.join(test_videos))

train_df = df[~val_mask].reset_index(drop=True)
val_df = df[val_mask].reset_index(drop=True)

# Zapis pomocniczych CSV w katalogu roboczym (/kaggle/working)
train_df.to_csv('train_labels_split.csv', index=False)
val_df.to_csv('val_labels_split.csv', index=False)

train_data = ClimbingDataset(
    csv_file='train_labels_split.csv',
    img_dir=IMG_DIR,
    transform=train_transforms
)

val_data = ClimbingDataset(
    csv_file='val_labels_split.csv',
    img_dir=IMG_DIR,
    transform=val_transforms
)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False)

# Inicjalizacja modelu
freeze = 6
model = get_climbing_model(freeze_until_block=freeze)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Podział zakończony! Urządzenie: {device}")
print(f"Próbki: Treningowe {len(train_data)}, Walidacyjne: {len(val_data)}")

Podział zakończony! Urządzenie: cuda
Próbki: Treningowe 2198, Walidacyjne: 390


In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-3)
#wybór schedulera
SCHEDULER_TYPE = "one_cycle" # "plateau", "cosine"
num_epochs = 40
patience = 6
counter = 0
best_val_loss = float('inf')
best_model_wts = copy.deepcopy(model.state_dict())
label_smoothing = 0.05


if SCHEDULER_TYPE == "one_cycle":
    total_steps = len(train_loader) * num_epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.001, pct_start=0.3, total_steps=total_steps)
    step_per_batch = True
elif SCHEDULER_TYPE == "plateau":
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=4)
    step_per_batch = False
elif SCHEDULER_TYPE == "cosine":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10)
    step_per_batch = False

#przygotowanie logów
history_logs = []
history_logs.append(f"{SCHEDULER_TYPE.upper()}\n")
history_logs.append(f"Blocks frozen: {freeze}, Start LR: {optimizer.param_groups[0]['lr']}, Weight Decay: {optimizer.param_groups[0]['weight_decay']}\n")

# parametry schedulera
sched_params = {k: v for k, v in scheduler.state_dict().items() if isinstance(v, (int, float, str, bool))}
history_logs.append(f"Scheduler Params: {sched_params}\n")
history_logs.append(f"Label Smoothing: {label_smoothing}, Patience: {patience}\n")
history_logs.append("-" * 60 + "\n")
history_logs.append("Epoch | LR | Train Loss | Val Loss\n")

MODEL_NAME = "phase2-one_cycle.pth"
LOG_NAME = f"{MODEL_NAME.split('.')[0]}.txt"

print(f"Start treningu na: {device}\n" + "-"*30)

for epoch in range(num_epochs):
    #trening
    model.train()
    running_train_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device).float()
        labels = labels * (1 - 2 * label_smoothing) + label_smoothing
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        if step_per_batch:
            scheduler.step()
        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    #walidacja
    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device).float()
            outputs = model(images)
            v_loss = criterion(outputs, labels)
            running_val_loss += v_loss.item()

    avg_val_loss = running_val_loss / len(val_loader)

    if not step_per_batch:
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(avg_val_loss)
        else:
            scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch [{epoch+1:02d}/{num_epochs}] | LR: {current_lr:.6f} | Loss: T {avg_train_loss:.4f} / V {avg_val_loss:.4f}")
    log_line = f"{epoch+1:02d} | {current_lr:.6f} | {avg_train_loss:.4f} | {avg_val_loss:.4f}\n"
    history_logs.append(log_line)

    if avg_val_loss < best_val_loss:
        print(f"Nowy najlepszy wynik")
        best_val_loss = avg_val_loss
        best_model_wts = copy.deepcopy(model.state_dict())
        torch.save(model.state_dict(), MODEL_NAME)
        counter = 0
    else:
        counter += 1
        if SCHEDULER_TYPE == "plateau":
            if counter >= patience:
                print(f"\nEARLY STOPPING po {epoch+1} epokach.")
                break

model.load_state_dict(best_model_wts)
print("-" * 30 + f"\nKoniec. Najlepszy wynik: {best_val_loss:.4f}")

Start treningu na: cuda
------------------------------
Epoch [01/40] | LR: 0.000056 | Loss: T 0.6139 / V 0.5655
Nowy najlepszy wynik
Epoch [02/40] | LR: 0.000104 | Loss: T 0.5205 / V 0.5290
Nowy najlepszy wynik


In [ ]:
# Folder na wyniki w katalogu roboczym Kaggle
SAVE_DIR = "/kaggle/working/checkpoints/proba1"
os.makedirs(SAVE_DIR, exist_ok=True)

# Przykład zapisu logów
with open(os.path.join(SAVE_DIR, f"{LOG_NAME}.txt"), "w") as f:
    f.writelines(history_logs)

# Przykład zapisu modelu
model_path = os.path.join(SAVE_DIR, f"{MODEL_NAME}.pth")
torch.save(model.state_dict(), model_path)

print(f"Wszystkie pliki zostały zapisane w: {SAVE_DIR}")